# Notebook 02 — Preprocesamiento

En este notebook transformamos el texto crudo en datos limpios
y estructurados listos para entrenar un modelo de Machine Learning.

El proceso sigue este orden:
1. Limpieza del texto
2. Tokenización y eliminación de stopwords
3. Lematización
4. División train/test
5. Vectorización con TF-IDF

In [11]:
# Librerías de manejo de datos
import pandas as pd
import numpy as np

# Librería para expresiones regulares (limpieza de texto)
import re

# Librería para NLP básico
import nltk
from nltk.corpus   import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem     import WordNetLemmatizer

# Descargar recursos de NLTK necesarios
nltk.download('stopwords',    quiet=True)
nltk.download('punkt',        quiet=True)
nltk.download('punkt_tab',    quiet=True)
nltk.download('wordnet',      quiet=True)

# Librería para dividir datos en train y test
from sklearn.model_selection import train_test_split

# Librería para vectorización
from sklearn.feature_extraction.text import TfidfVectorizer

# Librería para guardar objetos en disco
import joblib

# Librería para guardar matrices dispersas
import scipy.sparse as sp

# Cargamos el dataset original
df = pd.read_csv('../../data/raw/youtoxic_english_1000.csv')

# Comprobamos que cargó bien
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head(3)

Filas: 1000 | Columnas: 15


,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\r\nDont you reckon them 'black lives matter' ...,True,True,False,False,True,False,False,False,False,False,False,False


# 2. Limpieza del texto

Antes de tokenizar o vectorizar, necesitamos limpiar el texto crudo.
Construimos una función que aplica todos los pasos de limpieza
en el orden correcto.

Pasos:
1. Convertir a minúsculas
2. Eliminar URLs
3. Eliminar menciones (@usuario)
4. Eliminar hashtags (#tema)
5. Eliminar emojis
6. Eliminar puntuación y caracteres especiales
7. Eliminar espacios extra

In [12]:
def clean_text(text):
    """
    Limpia un texto aplicando los siguientes pasos:
    1. Minúsculas
    2. Eliminar URLs
    3. Eliminar menciones
    4. Eliminar hashtags
    5. Eliminar emojis y caracteres no ASCII
    6. Eliminar puntuación y caracteres especiales
    7. Eliminar espacios extra
    """
    # 1. Minúsculas
    text = text.lower()

    # 2. Eliminar URLs
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'www\S+',  '', text)

    # 3. Eliminar menciones
    text = re.sub(r'@\S+', '', text)

    # 4. Eliminar hashtags
    text = re.sub(r'#\S+', '', text)

    # 5. Eliminar emojis y caracteres no ASCII
    text = text.encode('ascii', 'ignore').decode('ascii')

    # 6. Eliminar puntuación y caracteres especiales
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 7. Eliminar espacios extra
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    return text

In [13]:
# Probamos con un ejemplo antes de aplicarla a todo el dataset
ejemplo = "Check this out!! 😡 http://youtube.com @usuario #BlackLivesMatter"
print("Antes:", ejemplo)
print("Después:", clean_text(ejemplo))

Antes: Check this out!! 😡 http://youtube.com @usuario #BlackLivesMatter
Después: check this out


**Verificación:**
La función de limpieza funciona correctamente.

- Convierte a minúsculas ✓
- Elimina URLs ✓
- Elimina menciones ✓
- Elimina hashtags ✓
- Elimina emojis ✓
- Elimina puntuación ✓

El texto resultante contiene solo palabras en minúsculas
sin ruido. Listo para tokenizar.

# 3. Tokenización y eliminación de stopwords

Una vez limpio el texto, lo dividimos en palabras individuales
y eliminamos las palabras vacías que no aportan significado.

Pasos:
1. Tokenizar: dividir el texto en palabras
2. Eliminar stopwords: quitar palabras vacías en inglés

In [14]:
# Cargamos las stopwords en inglés
stop_words = set(stopwords.words('english'))

def tokenize_and_remove_stopwords(text):
    """
    Tokeniza el texto y elimina stopwords y palabras cortas.
    Devuelve el texto como string limpio.
    """
    # Tokenizamos
    tokens = word_tokenize(text)

    # Filtramos stopwords y palabras muy cortas
    tokens = [
        token for token in tokens
        if token not in stop_words
        and len(token) > 2
    ]

    # Devolvemos como texto
    return ' '.join(tokens)

In [15]:
# Probamos con el resultado del bloque anterior
ejemplo_limpio = clean_text(
    "Black people are getting shot by police officers every day"
)
print("Después de limpiar:", ejemplo_limpio)
print("Después de tokenizar:", tokenize_and_remove_stopwords(ejemplo_limpio))

Después de limpiar: black people are getting shot by police officers every day
Después de tokenizar: black people getting shot police officers every day


**Verificación:**
La tokenización y eliminación de stopwords funciona correctamente.

- Tokenización: el texto se divide en palabras individuales ✓
- Stopwords eliminadas: are, by, every ✓
- Palabras con significado conservadas: black, people,
  shot, police, officers ✓

Las palabras clave para detectar toxicidad se mantienen.
Listo para lematizar.

# 4. Lematización

La lematización reduce cada palabra a su forma base (lema).
Esto agrupa variantes de la misma palabra para que el modelo
las trate como una sola cosa.

Ejemplos:
- running, runs, ran  → run
- officers, officer   → officer
- getting, gets, got  → get
- people, person      → person

In [16]:
# Creamos el lematizador
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    """
    Lematiza cada palabra del texto reduciéndola a su forma base.
    Intenta primero como verbo y si no cambia, como sustantivo.
    """
    tokens = []

    for token in text.split():
        # Intentamos como verbo primero
        lemma_v = lemmatizer.lemmatize(token, pos='v')
        if lemma_v != token:
            tokens.append(lemma_v)
        else:
            tokens.append(lemmatizer.lemmatize(token))

    return ' '.join(tokens)

In [17]:
# Encadenamos los tres pasos
ejemplo = "Black people are getting shot by police officers every day"

paso1 = clean_text(ejemplo)
paso2 = tokenize_and_remove_stopwords(paso1)
paso3 = lemmatize_text(paso2)

print("Original: ", ejemplo)
print("Limpio:   ", paso1)
print("Tokenizado:", paso2)
print("Lematizado:", paso3)

Original:  Black people are getting shot by police officers every day
Limpio:    black people are getting shot by police officers every day
Tokenizado: black people getting shot police officers every day
Lematizado: black people get shoot police officer every day


**Verificación:**
La lematización funciona correctamente.

Cambios aplicados:
- getting → get (verbo a forma base) ✓
- officers → officer (plural a singular) ✓
- shot → shoot (pasado a infinitivo) ✓

Nota: "shot" se lematiza como "shoot" porque el lematizador
lo interpreta como verbo. Es una limitación del enfoque
clásico sin contexto. BERT resolvería este caso correctamente.

El texto está listo para aplicar el pipeline completo
al dataset.

# 5. Aplicar el pipeline de preprocesamiento al dataset

Ahora aplicamos las tres funciones definidas anteriormente:
1. Limpieza del texto
2. Tokenización + eliminación de stopwords
3. Lematización

El resultado será una nueva columna llamada 'Text_clean'
que contiene el texto completamente procesado y listo
para vectorizar.


In [ ]:
# Aplicamos la limpieza
df['Text_clean'] = df['Text'].apply(clean_text)

# Aplicamos tokenización + stopwords
df['Text_clean'] = df['Text_clean'].apply(tokenize_and_remove_stopwords)

# Aplicamos lematización
df['Text_clean'] = df['Text_clean'].apply(lemmatize_text)

df[['Text', 'Text_clean']].head(10)
